In [22]:

## 1 · Environment Setup


# ── Install dependencies (run once) ──────────────────────────────────────────
import subprocess, sys

_PACKAGES = [
    "torch", "torchaudio",
    "transformers>=4.40", "accelerate>=0.28",
    "peft>=0.10",
    "librosa", "soundfile",
]

subprocess.check_call(
    [sys.executable, "-m", "pip", "install", "-q", "--upgrade"] + _PACKAGES,
)

# bitsandbytes: install separately; 0.44.1+ supports Python 3.12 + CUDA 12.
# Never reload after first import — re-registering PyTorch dispatch kernels
# raises a "kernel already registered" error on subsequent cell runs.
subprocess.check_call(
    [sys.executable, "-m", "pip", "install", "-q", "bitsandbytes>=0.44.1"],
)

try:
    # Use cached module if already imported to avoid kernel re-registration
    import sys as _sys
    if "bitsandbytes" not in _sys.modules:
        import bitsandbytes as bnb
    else:
        bnb = _sys.modules["bitsandbytes"]
    print(f"✓ bitsandbytes {bnb.__version__} ready")
    BNB_AVAILABLE = True
except Exception as e:
    print(f"⚠ bitsandbytes not available ({e}) — will load model in float16")
    BNB_AVAILABLE = False

print("✓ packages installed")


✓ bitsandbytes 0.49.2 ready
✓ packages installed


## 2 · Imports & Configuration

In [31]:
import os
import csv
import json
import math
import random
import re
import shutil
import urllib.request
import warnings
import zipfile
from pathlib import Path
from dataclasses import dataclass, asdict
from typing import List, Dict, Optional, Tuple

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchaudio
import soundfile as sf

from transformers import (
    Wav2Vec2Model,
    Wav2Vec2Processor,
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
)
from peft import LoraConfig, get_peft_model, TaskType, prepare_model_for_kbit_training

warnings.filterwarnings("ignore", category=FutureWarning)

# ── Reproducibility ──────────────────────────────────────────────────────────
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

torch.backends.cuda.matmul.allow_tf32 = True
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device : {DEVICE}")
if DEVICE.type == "cuda":
    print(f"GPU    : {torch.cuda.get_device_name(0)}")
    print(f"VRAM   : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

# ── Hyperparameters (single place to edit) ───────────────────────────────────
CFG = {
    # audio
    "audio_encoder_name": "facebook/wav2vec2-base-960h",
    "target_sr": 16_000,
    # language model
    "lm_name": "microsoft/phi-2",          # 2.7 B — fits Colab T4 in 4-bit
    "max_seq_len": 512,
    # LoRA
    "lora_r": 16,
    "lora_alpha": 32,
    "lora_dropout": 0.05,
    # training
    "lr": 1e-5,
    "weight_decay": 0.01,
    "num_epochs": 8,
    "grad_clip": 1.0,
    # inference
    "temperature": 0.2,
    "top_p": 0.9,
    "max_new_tokens": 256,
    # evaluation
    "eval_runs": 3,
    "eval_temperature": 0.0,
    "eval_top_p": 1.0,
    "eval_semantic_match": True,
    # ESC-50 evaluation runtime controls
    "esc50_eval_limit": 80,
    "esc50_eval_max_new_tokens": 24,
    # ESC-50 hybrid dataset
    "esc50_root": "./esc50",
    "esc50_use": True,
    "esc50_auto_download": True,
    "esc50_download_url": "https://github.com/karolpiczak/ESC-50/archive/refs/heads/master.zip",
    "esc50_train_folds": [1, 2, 3, 4],
    "esc50_val_folds": [5],
    "esc50_train_limit": 0,
    "esc50_val_limit": 0,
    "esc50_mix_ratio": 1.0,
    # paths
    "data_dir": "./synthetic_data",
    "output_path": "./results.jsonl",
    "ckpt_dir": "./checkpoints",
}

# Resolve paths once so files are written where the notebook is executed.
PROJECT_ROOT = Path.cwd().resolve()
CFG["data_dir"] = str((PROJECT_ROOT / CFG["data_dir"]).resolve())
CFG["output_path"] = str((PROJECT_ROOT / CFG["output_path"]).resolve())
CFG["ckpt_dir"] = str((PROJECT_ROOT / CFG["ckpt_dir"]).resolve())
CFG["esc50_root"] = str((PROJECT_ROOT / CFG["esc50_root"]).resolve())
print(f"Data dir   : {CFG['data_dir']}")
print(f"Output file: {CFG['output_path']}")
print(f"CKPT dir   : {CFG['ckpt_dir']}")
print(f"ESC-50 dir : {CFG['esc50_root']}")

Device : cuda
GPU    : Tesla T4
VRAM   : 15.6 GB
Data dir   : /content/synthetic_data
Output file: /content/results.jsonl
CKPT dir   : /content/checkpoints
ESC-50 dir : /content/esc50


## 3 · Audio Frontend (frozen wav2vec2)

In [24]:
class AudioEncoder(nn.Module):
    """
    Frozen wav2vec2 encoder.
    Input  : path to a .wav file (any sample-rate, any channels)
    Output : single audio vector of shape (1, hidden_size=768)
    """

    def __init__(self, model_name: str = CFG["audio_encoder_name"]):
        super().__init__()
        self.processor = Wav2Vec2Processor.from_pretrained(model_name)
        self.encoder = Wav2Vec2Model.from_pretrained(model_name)

        # Freeze every parameter
        for p in self.encoder.parameters():
            p.requires_grad = False
        self.encoder.eval()

        self.hidden_size: int = self.encoder.config.hidden_size  # 768

    # ── helpers ───────────────────────────────────────────────────────────
    @staticmethod
    def load_audio(path: str, target_sr: int = CFG["target_sr"]) -> torch.Tensor:
        """Load audio file → mono float32 tensor at target_sr."""
        waveform, sr = torchaudio.load(path)
        if waveform.shape[0] > 1:
            waveform = waveform.mean(dim=0, keepdim=True)
        if sr != target_sr:
            waveform = torchaudio.transforms.Resample(sr, target_sr)(waveform)
        return waveform.squeeze(0)  # (num_samples,)

    # ── forward ──────────────────────────────────────────────────────────
    @torch.no_grad()
    def encode_sequence(self, audio_path: str) -> torch.Tensor:
        """Return frame-level hidden states of shape (1, T, hidden_size)."""
        waveform = self.load_audio(audio_path)

        device = next(self.encoder.parameters()).device
        inputs = self.processor(
            waveform.numpy(),
            sampling_rate=CFG["target_sr"],
            return_tensors="pt",
            padding=True,
        )
        inputs = {k: v.to(device) for k, v in inputs.items()}
        hidden = self.encoder(**inputs).last_hidden_state
        return hidden

    @torch.no_grad()
    def forward(self, audio_path: str) -> torch.Tensor:
        hidden = self.encode_sequence(audio_path)
        pooled = hidden.mean(dim=1)                                # (1, 768)
        return pooled


# Quick smoke test
_ae = AudioEncoder()
print(f"✓ AudioEncoder  |  hidden_size = {_ae.hidden_size}")
del _ae


Loading weights:   0%|          | 0/210 [00:00<?, ?it/s]

Wav2Vec2Model LOAD REPORT from: facebook/wav2vec2-base-960h
Key               | Status     | 
------------------+------------+-
lm_head.weight    | UNEXPECTED | 
lm_head.bias      | UNEXPECTED | 
masked_spec_embed | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


✓ AudioEncoder  |  hidden_size = 768


## 4 · Dataset Format & Synthetic Data

In [25]:
@dataclass
class AudioSample:
    """One audio-reasoning example."""
    audio_path: str
    question: str
    reasoning: str
    answer: str


def _tone(freq: float, dur: float, sr: int, amp: float = 0.4) -> np.ndarray:
    t = np.linspace(0, dur, int(sr * dur), endpoint=False, dtype=np.float32)
    return (amp * np.sin(2 * np.pi * freq * t)).astype(np.float32)


def _chirp(f0: float, f1: float, dur: float, sr: int, amp: float = 0.4) -> np.ndarray:
    t = np.linspace(0, dur, int(sr * dur), endpoint=False, dtype=np.float32)
    k = (f1 - f0) / max(dur, 1e-6)
    phase = 2 * np.pi * (f0 * t + 0.5 * k * t * t)
    return (amp * np.sin(phase)).astype(np.float32)


def _silence(dur: float, sr: int) -> np.ndarray:
    return np.zeros(int(sr * dur), dtype=np.float32)


def _mix(*tracks: np.ndarray) -> np.ndarray:
    max_len = max(len(x) for x in tracks)
    out = np.zeros(max_len, dtype=np.float32)
    for tr in tracks:
        out[: len(tr)] += tr
    out = np.clip(out, -1.0, 1.0)
    return out.astype(np.float32)


def _concat(parts: List[np.ndarray]) -> np.ndarray:
    return np.concatenate(parts).astype(np.float32)


def create_synthetic_dataset(out_dir: str = CFG["data_dir"]) -> List[AudioSample]:
    """Create temporal audio scenes (sequence, overlap, pauses, chirps) with QA."""
    out_path = Path(out_dir).expanduser().resolve()
    out_path.mkdir(parents=True, exist_ok=True)
    sr = CFG["target_sr"]

    scene_specs = [
        {
            "name": "scene_01_two_tones_low_high",
            "wav": _concat([_tone(220, 0.8, sr), _silence(0.2, sr), _tone(880, 0.8, sr)]),
            "q": "Does pitch increase or decrease over time?",
            "r": "The first event is low frequency (~220 Hz), then after a short gap a much higher tone (~880 Hz) appears. The sequence moves from low to high.",
            "a": "Pitch increases over time.",
        },
        {
            "name": "scene_02_two_tones_high_low",
            "wav": _concat([_tone(900, 0.7, sr), _silence(0.2, sr), _tone(250, 0.9, sr)]),
            "q": "Does pitch increase or decrease over time?",
            "r": "The first tone is high (~900 Hz) and the second is lower (~250 Hz), so the sequence goes downward in pitch.",
            "a": "Pitch decreases over time.",
        },
        {
            "name": "scene_03_chirp_up",
            "wav": _chirp(200, 1200, 1.6, sr),
            "q": "Is this an ascending or descending sweep?",
            "r": "The frequency continuously moves upward from about 200 Hz to around 1200 Hz.",
            "a": "Ascending sweep.",
        },
        {
            "name": "scene_04_chirp_down",
            "wav": _chirp(1400, 250, 1.6, sr),
            "q": "Is this an ascending or descending sweep?",
            "r": "The chirp starts high and continuously drops to a much lower frequency.",
            "a": "Descending sweep.",
        },
        {
            "name": "scene_05_three_events",
            "wav": _concat([
                _tone(300, 0.45, sr), _silence(0.2, sr),
                _tone(600, 0.45, sr), _silence(0.2, sr),
                _tone(900, 0.45, sr),
            ]),
            "q": "How many distinct tone events are present?",
            "r": "There are three separated tone bursts with short silent gaps between them.",
            "a": "Three tone events.",
        },
        {
            "name": "scene_06_long_then_short",
            "wav": _concat([_tone(500, 1.6, sr), _silence(0.2, sr), _tone(500, 0.5, sr)]),
            "q": "Is the first tone longer than the second?",
            "r": "The first tone lasts about 1.6 s while the second lasts about 0.5 s.",
            "a": "Yes, the first tone is longer.",
        },
        {
            "name": "scene_07_short_then_long",
            "wav": _concat([_tone(450, 0.4, sr), _silence(0.2, sr), _tone(450, 1.4, sr)]),
            "q": "Is the first tone longer than the second?",
            "r": "The first tone is short (~0.4 s), and the second is much longer (~1.4 s).",
            "a": "No, the first tone is shorter.",
        },
        {
            "name": "scene_08_overlap_two_tones",
            "wav": _mix(_tone(330, 1.4, sr, 0.25), _concat([_silence(0.35, sr), _tone(880, 1.05, sr, 0.25)])),
            "q": "Is there a period where two tones are sounding at the same time?",
            "r": "One tone starts first and another starts before the first ends, creating overlap.",
            "a": "Yes, there is an overlapping segment.",
        },
        {
            "name": "scene_09_gap_detection",
            "wav": _concat([_tone(700, 0.7, sr), _silence(0.6, sr), _tone(700, 0.7, sr)]),
            "q": "Is there a noticeable silent gap between two tone events?",
            "r": "Two similar tones are separated by about 0.6 seconds of silence.",
            "a": "Yes, there is a clear silent gap.",
        },
        {
            "name": "scene_10_constant_mid",
            "wav": _tone(1000, 1.2, sr),
            "q": "What frequency range best describes this tone?",
            "r": "A steady single-frequency tone around 1 kHz corresponds to a mid-to-high frequency range.",
            "a": "Mid-to-high frequency range.",
        },
        {
            "name": "scene_11_order_question",
            "wav": _concat([_tone(260, 0.55, sr), _silence(0.2, sr), _tone(740, 0.55, sr)]),
            "q": "Did the lower-pitched event occur before the higher-pitched one?",
            "r": "The first event is about 260 Hz and the second around 740 Hz.",
            "a": "Yes, low pitch occurs first.",
        },
        {
            "name": "scene_12_order_question_reverse",
            "wav": _concat([_tone(760, 0.55, sr), _silence(0.2, sr), _tone(260, 0.55, sr)]),
            "q": "Did the lower-pitched event occur before the higher-pitched one?",
            "r": "The first event is high (~760 Hz), followed by low (~260 Hz).",
            "a": "No, the higher pitch occurs first.",
        },
    ]

    samples: List[AudioSample] = []
    for s in scene_specs:
        path = out_path / f"{s['name']}.wav"
        sf.write(str(path), s["wav"], sr)
        if not path.exists():
            raise FileNotFoundError(f"Failed to write {path}")
        samples.append(AudioSample(str(path), s["q"], s["r"], s["a"]))

    with open(out_path / "metadata.json", "w", encoding="utf-8") as f:
        json.dump([asdict(s) for s in samples], f, indent=2)

    print(f"✓ Created {len(samples)} temporal-scene samples in {out_path}")
    return samples


def create_mmar_benchmark(out_dir: Optional[str] = None) -> List[AudioSample]:
    """
    MMAR-style benchmark split focused on temporal audio reasoning.
    If you have an official MMAR file, replace this generator with that loader.
    """
    base = Path(CFG["data_dir"]).resolve()
    bench_dir = Path(out_dir).resolve() if out_dir else (base / "mmar_benchmark")
    bench_dir.mkdir(parents=True, exist_ok=True)
    sr = CFG["target_sr"]

    bench_specs = [
        {
            "name": "mmar_01_temporal_order",
            "wav": _concat([_tone(240, 0.5, sr), _silence(0.15, sr), _tone(960, 0.5, sr)]),
            "q": "In this temporal scene, which comes first: low pitch or high pitch?",
            "r": "The scene starts with a low-frequency event then transitions to a higher-frequency event.",
            "a": "Low pitch comes first.",
            "tag": "temporal_order",
        },
        {
            "name": "mmar_02_event_count",
            "wav": _concat([_tone(400, 0.35, sr), _silence(0.18, sr), _tone(600, 0.35, sr), _silence(0.18, sr), _tone(800, 0.35, sr)]),
            "q": "How many discrete events are in the scene?",
            "r": "Three bursts are separated by silent intervals.",
            "a": "Three events.",
            "tag": "event_count",
        },
        {
            "name": "mmar_03_duration_compare",
            "wav": _concat([_tone(520, 1.2, sr), _silence(0.2, sr), _tone(520, 0.45, sr)]),
            "q": "Is the first event longer than the second event?",
            "r": "The first event lasts significantly longer than the second.",
            "a": "Yes, the first event is longer.",
            "tag": "duration_compare",
        },
        {
            "name": "mmar_04_overlap_detect",
            "wav": _mix(_tone(300, 1.3, sr, 0.24), _concat([_silence(0.4, sr), _tone(700, 0.9, sr, 0.24)])),
            "q": "Do any events overlap in time?",
            "r": "The later event starts before the earlier event ends.",
            "a": "Yes, events overlap.",
            "tag": "overlap",
        },
        {
            "name": "mmar_05_trend",
            "wav": _chirp(1200, 250, 1.5, sr),
            "q": "What is the overall frequency trend across time?",
            "r": "Frequency decreases from high to low over the clip.",
            "a": "Decreasing trend.",
            "tag": "trend",
        },
        {
            "name": "mmar_06_pause_detect",
            "wav": _concat([_tone(760, 0.6, sr), _silence(0.55, sr), _tone(760, 0.6, sr)]),
            "q": "Is there a significant pause separating the two events?",
            "r": "A long silence separates two similar events.",
            "a": "Yes, there is a significant pause.",
            "tag": "pause",
        },
    ]

    samples: List[AudioSample] = []
    for s in bench_specs:
        path = bench_dir / f"{s['name']}.wav"
        sf.write(str(path), s["wav"], sr)
        samples.append(AudioSample(str(path), s["q"], s["r"], s["a"]))

    metadata_specs = [{k: v for k, v in s.items() if k != "wav"} for s in bench_specs]
    with open(bench_dir / "metadata.json", "w", encoding="utf-8") as f:
        json.dump(metadata_specs, f, indent=2)

    print(f"✓ Created MMAR-style benchmark ({len(samples)} samples) in {bench_dir}")
    return samples


def _esc50_question() -> str:
    return "What is the primary environmental sound class in this clip?"


def _esc50_reasoning(category: str) -> str:
    return f"The clip contains acoustic patterns most consistent with the '{category}' class."


def ensure_esc50_dataset(
    root_dir: str = CFG["esc50_root"],
    auto_download: bool = True,
    download_url: str = "https://github.com/karolpiczak/ESC-50/archive/refs/heads/master.zip",
) -> bool:
    """Ensure ESC-50 exists in expected structure (<root>/meta/esc50.csv, <root>/audio)."""
    root = Path(root_dir).expanduser().resolve()
    csv_path = root / "meta" / "esc50.csv"
    audio_dir = root / "audio"
    if csv_path.exists() and audio_dir.exists():
        return True

    if not auto_download:
        return False

    root.parent.mkdir(parents=True, exist_ok=True)
    zip_path = root.parent / "esc50_download.zip"

    try:
        print(f"↻ Downloading ESC-50 from {download_url}")
        urllib.request.urlretrieve(download_url, zip_path)

        extract_dir = root.parent / "esc50_extract"
        if extract_dir.exists():
            shutil.rmtree(extract_dir)
        extract_dir.mkdir(parents=True, exist_ok=True)

        with zipfile.ZipFile(zip_path, "r") as zf:
            zf.extractall(extract_dir)

        candidates = [extract_dir] + [p for p in extract_dir.glob("**/*") if p.is_dir()]
        source_root = None
        for cand in candidates:
            if (cand / "meta" / "esc50.csv").exists() and (cand / "audio").exists():
                source_root = cand
                break

        if source_root is None:
            print("⚠ Downloaded archive, but could not locate ESC-50 meta/audio folders.")
            return False

        if root.exists():
            shutil.rmtree(root)
        shutil.copytree(source_root, root)
        print(f"✓ ESC-50 prepared at {root}")
        return (root / "meta" / "esc50.csv").exists() and (root / "audio").exists()
    except Exception as e:
        print(f"⚠ ESC-50 auto-download failed: {e}")
        return False
    finally:
        if zip_path.exists():
            zip_path.unlink(missing_ok=True)
        tmp_extract = root.parent / "esc50_extract"
        if tmp_extract.exists():
            shutil.rmtree(tmp_extract, ignore_errors=True)


def load_esc50_hybrid_dataset(
    root_dir: str = CFG["esc50_root"],
    train_folds: Optional[List[int]] = None,
    val_folds: Optional[List[int]] = None,
    train_limit: int = 0,
    val_limit: int = 0,
    auto_download: bool = True,
    download_url: str = "https://github.com/karolpiczak/ESC-50/archive/refs/heads/master.zip",
) -> Tuple[List[AudioSample], List[AudioSample], List[str]]:
    """Load ESC-50 metadata and convert entries into AudioSample format."""
    ensure_esc50_dataset(
        root_dir=root_dir,
        auto_download=auto_download,
        download_url=download_url,
    )

    root = Path(root_dir).expanduser().resolve()
    csv_path = root / "meta" / "esc50.csv"
    audio_dir = root / "audio"

    if not csv_path.exists() or not audio_dir.exists():
        print(f"⚠ ESC-50 not found at {root}")
        print("  Expected structure: <esc50_root>/meta/esc50.csv and <esc50_root>/audio/*.wav")
        return [], [], []

    train_fold_set = set(train_folds or CFG.get("esc50_train_folds", [1, 2, 3, 4]))
    val_fold_set = set(val_folds or CFG.get("esc50_val_folds", [5]))

    train_samples: List[AudioSample] = []
    val_samples: List[AudioSample] = []
    labels = set()

    with open(csv_path, encoding="utf-8") as f:
        reader = csv.DictReader(f)
        for row in reader:
            fold = int(row["fold"])
            category = row["category"].strip().lower()
            wav_path = (audio_dir / row["filename"]).resolve()
            if not wav_path.exists():
                continue

            labels.add(category)
            sample = AudioSample(
                audio_path=str(wav_path),
                question=_esc50_question(),
                reasoning=_esc50_reasoning(category),
                answer=category,
            )

            if fold in train_fold_set:
                train_samples.append(sample)
            elif fold in val_fold_set:
                val_samples.append(sample)

    if train_limit and train_limit > 0:
        random.shuffle(train_samples)
        train_samples = train_samples[:train_limit]
    if val_limit and val_limit > 0:
        random.shuffle(val_samples)
        val_samples = val_samples[:val_limit]

    label_list = sorted(labels)
    print(f"✓ Loaded ESC-50 train={len(train_samples)} val={len(val_samples)} labels={len(label_list)} from {root}")
    return train_samples, val_samples, label_list


def build_hybrid_train_dataset(
    temporal_samples: List[AudioSample],
    esc50_samples: List[AudioSample],
    esc50_mix_ratio: float = 1.0,
) -> List[AudioSample]:
    """Mix temporal QA samples with ESC-50 samples by ratio to improve audio grounding."""
    base = list(temporal_samples)
    if len(esc50_samples) == 0 or esc50_mix_ratio <= 0:
        return base

    target_esc = max(1, int(len(base) * float(esc50_mix_ratio)))
    if len(esc50_samples) >= target_esc:
        esc_subset = random.sample(esc50_samples, target_esc)
    else:
        esc_subset = random.choices(esc50_samples, k=target_esc)

    mixed = base + esc_subset
    random.shuffle(mixed)
    return mixed


dataset = create_synthetic_dataset()
mmar_benchmark = create_mmar_benchmark()

esc50_train: List[AudioSample] = []
esc50_val: List[AudioSample] = []
ESC50_LABELS: List[str] = []
if CFG.get("esc50_use", True):
    esc50_train, esc50_val, ESC50_LABELS = load_esc50_hybrid_dataset(
        root_dir=CFG["esc50_root"],
        train_folds=CFG.get("esc50_train_folds", [1, 2, 3, 4]),
        val_folds=CFG.get("esc50_val_folds", [5]),
        train_limit=int(CFG.get("esc50_train_limit", 0)),
        val_limit=int(CFG.get("esc50_val_limit", 0)),
        auto_download=bool(CFG.get("esc50_auto_download", True)),
        download_url=CFG.get("esc50_download_url", "https://github.com/karolpiczak/ESC-50/archive/refs/heads/master.zip"),
    )

train_dataset = build_hybrid_train_dataset(
    temporal_samples=dataset,
    esc50_samples=esc50_train,
    esc50_mix_ratio=float(CFG.get("esc50_mix_ratio", 1.0)),
)

for s in dataset[:6]:
    print(f"  {Path(s.audio_path).name:28s}  Q: {s.question[:65]}")
print(f"... total train scenes: {len(dataset)}")
print(f"... total MMAR benchmark scenes: {len(mmar_benchmark)}")
print(f"... ESC-50 train samples: {len(esc50_train)}")
print(f"... ESC-50 val samples: {len(esc50_val)}")
print(f"... hybrid train samples: {len(train_dataset)}")

✓ Created 12 temporal-scene samples in /content/synthetic_data
✓ Created MMAR-style benchmark (6 samples) in /content/synthetic_data/mmar_benchmark
↻ Downloading ESC-50 from https://github.com/karolpiczak/ESC-50/archive/refs/heads/master.zip
✓ ESC-50 prepared at /content/esc50
✓ Loaded ESC-50 train=1600 val=400 labels=50 from /content/esc50
  scene_01_two_tones_low_high.wav  Q: Does pitch increase or decrease over time?
  scene_02_two_tones_high_low.wav  Q: Does pitch increase or decrease over time?
  scene_03_chirp_up.wav         Q: Is this an ascending or descending sweep?
  scene_04_chirp_down.wav       Q: Is this an ascending or descending sweep?
  scene_05_three_events.wav     Q: How many distinct tone events are present?
  scene_06_long_then_short.wav  Q: Is the first tone longer than the second?
... total train scenes: 12
... total MMAR benchmark scenes: 6
... ESC-50 train samples: 1600
... ESC-50 val samples: 400
... hybrid train samples: 24


## 5 · Language Model + LoRA

In [32]:
import os
import csv
import json
import math
import random
import re
import shutil
import urllib.request
import warnings
import zipfile
from pathlib import Path
from dataclasses import dataclass, asdict
from typing import List, Dict, Optional, Tuple

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchaudio
import soundfile as sf

from transformers import (
    Wav2Vec2Model,
    Wav2Vec2Processor,
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    AutoConfig,
)
from peft import LoraConfig, get_peft_model, TaskType, prepare_model_for_kbit_training

warnings.filterwarnings("ignore", category=FutureWarning)

# ── Reproducibility ──────────────────────────────────────────────────────────
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

torch.backends.cuda.matmul.allow_tf32 = True
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device : {DEVICE}")
if DEVICE.type == "cuda":
    print(f"GPU    : {torch.cuda.get_device_name(0)}")
    print(f"VRAM   : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

# Check bitsandbytes availability — catch any import-time crash, not just ImportError
try:
    import bitsandbytes as _bnb
    BNB_AVAILABLE = True
except Exception:
    BNB_AVAILABLE = False

# ── Hyperparameters (single place to edit) ───────────────────────────────────
CFG = {
    # audio
    "audio_encoder_name": "facebook/wav2vec2-base-960h",
    "target_sr": 16_000,
    # language model
    "lm_name": "microsoft/phi-2",          # 2.7 B — fits Colab T4 in 4-bit
    "max_seq_len": 512,
    # LoRA
    "lora_r": 16,
    "lora_alpha": 32,
    "lora_dropout": 0.05,
    # training
    "lr": 1e-5,
    "weight_decay": 0.01,
    "num_epochs": 8,
    "grad_clip": 1.0,
    # inference
    "temperature": 0.2,
    "top_p": 0.9,
    "max_new_tokens": 256,
    # evaluation
    "eval_runs": 3,
    "eval_temperature": CFG.get("eval_temperature", 0.0),
    "eval_top_p": CFG.get("eval_top_p", 1.0),
    "eval_semantic_match": CFG.get("eval_semantic_match", True),
    # ESC-50 evaluation runtime controls
    "esc50_eval_limit": CFG.get("esc50_eval_limit", 80),
    "esc50_eval_max_new_tokens": CFG.get("esc50_eval_max_new_tokens", 24),
    # audio-prefix grounding
    "audio_prefix_tokens": 6,
    "audio_prefix_scale": 0.08,
    # supervision shaping
    "answer_loss_weight": 2.5,
    # contrastive negatives
    "neg_pair_prob": 0.5,
    "contrastive_weight": 0.35,
    "contrastive_margin": 0.25,
    # curriculum
    "curriculum_warmup_epochs": 3,
    # ESC-50 hybrid dataset
    "esc50_root": CFG["esc50_root"],
    "esc50_use": CFG.get("esc50_use", True),
    "esc50_auto_download": CFG.get("esc50_auto_download", True),
    "esc50_download_url": CFG.get("esc50_download_url", "https://github.com/karolpiczak/ESC-50/archive/refs/heads/master.zip"),
    "esc50_train_folds": CFG.get("esc50_train_folds", [1, 2, 3, 4]),
    "esc50_val_folds": CFG.get("esc50_val_folds", [5]),
    "esc50_train_limit": CFG.get("esc50_train_limit", 0),
    "esc50_val_limit": CFG.get("esc50_val_limit", 0),
    "esc50_mix_ratio": CFG.get("esc50_mix_ratio", 1.0),
    # paths
    "data_dir": CFG["data_dir"],
    "output_path": CFG["output_path"],
    "ckpt_dir": CFG["ckpt_dir"],
}


def _find_linear_names(model: nn.Module) -> List[str]:
    """Return unique short names of all nn.Linear layers (excluding lm_head)."""
    names = set()
    for n, m in model.named_modules():
        if isinstance(m, nn.Linear):
            short = n.split(".")[-1]
            if short != "lm_head":
                names.add(short)
    return sorted(names)


def load_language_model(
    model_name: str = CFG["lm_name"],
) -> Tuple[nn.Module, AutoTokenizer]:
    """
    Load a causal LM and wrap it with LoRA.
    Uses 4-bit NF4 quantization when bitsandbytes is available;
    falls back to float16 otherwise (needs more VRAM).
    """
    # ── Tokenizer ────────────────────────────────────────────────────────
    tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    if tokenizer.pad_token_id is None:
        tokenizer.pad_token_id = tokenizer.eos_token_id

    # ── Model config ─────────────────────────────────────────────────────
    model_config = AutoConfig.from_pretrained(model_name, trust_remote_code=True)
    model_config.pad_token_id = tokenizer.pad_token_id

    # ── Load model (quantized or plain float16) ───────────────────────────
    if BNB_AVAILABLE and DEVICE.type == "cuda":
        print("  Loading in 4-bit NF4 (bitsandbytes)…")
        bnb_cfg = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.float16,  # float16 for T4 compatibility
            bnb_4bit_use_double_quant=True,
        )
        model = AutoModelForCausalLM.from_pretrained(
            model_name,
            config=model_config,
            quantization_config=bnb_cfg,
            device_map="auto",
            trust_remote_code=True,
        )
        model = prepare_model_for_kbit_training(model)
    else:
        print("  Loading in float16 (no 4-bit quantization)…")
        model = AutoModelForCausalLM.from_pretrained(
            model_name,
            config=model_config,
            torch_dtype=torch.float16,
            device_map="auto",
            trust_remote_code=True,
            low_cpu_mem_usage=True,
        )

    model.config.use_cache = False
    if hasattr(model, "gradient_checkpointing_enable"):
        model.gradient_checkpointing_enable()

    # ── LoRA ─────────────────────────────────────────────────────────────
    target_modules = _find_linear_names(model)
    print(f"  LoRA target modules: {target_modules}")

    lora_cfg = LoraConfig(
        task_type=TaskType.CAUSAL_LM,
        r=CFG["lora_r"],
        lora_alpha=CFG["lora_alpha"],
        lora_dropout=CFG["lora_dropout"],
        target_modules=target_modules,
        bias="none",
    )
    model = get_peft_model(model, lora_cfg)
    model.print_trainable_parameters()

    return model, tokenizer


lm_model, tokenizer = load_language_model()
LM_DIM: int = lm_model.config.hidden_size
print(f"✓ LM loaded  |  hidden_size = {LM_DIM}")

Device : cuda
GPU    : Tesla T4
VRAM   : 15.6 GB
  Loading in 4-bit NF4 (bitsandbytes)…


Loading weights:   0%|          | 0/453 [00:00<?, ?it/s]

  LoRA target modules: ['dense', 'fc1', 'fc2', 'k_proj', 'q_proj', 'v_proj']
trainable params: 23,592,960 || all params: 2,803,276,800 || trainable%: 0.8416
✓ LM loaded  |  hidden_size = 2560


## 6 · Audio–Text Fusion Model

In [27]:
class AudioLanguageModel(nn.Module):
    """
    Fuses a frozen audio encoder with a LoRA-adapted LLM via a trainable
    projection that maps temporal audio chunks into multiple virtual
    prefix tokens in the LLM embedding space.
    """

    def __init__(
        self,
        audio_encoder: AudioEncoder,
        lm_model: nn.Module,
        lm_tokenizer: AutoTokenizer,
    ):
        super().__init__()
        self.audio_encoder = audio_encoder
        self.lm = lm_model
        self.tok = lm_tokenizer

        audio_dim = audio_encoder.hidden_size             # 768
        lm_dim = lm_model.config.hidden_size              # e.g. 2560
        self.audio_prefix_tokens = int(CFG.get("audio_prefix_tokens", 6))

        embed_layer = lm_model.get_input_embeddings()
        self._embed_device = next(embed_layer.parameters()).device
        self._embed_dtype = next(embed_layer.parameters()).dtype
        print(f"  LLM embed device: {self._embed_device}, dtype: {self._embed_dtype}")

        self.audio_encoder = self.audio_encoder.to(self._embed_device)

        self.audio_proj = nn.Sequential(
            nn.Linear(audio_dim, lm_dim, bias=True),
            nn.GELU(),
            nn.Linear(lm_dim, lm_dim, bias=True),
            nn.LayerNorm(lm_dim),
        )
        for layer in self.audio_proj:
            if isinstance(layer, nn.Linear):
                nn.init.xavier_normal_(layer.weight, gain=0.05)
                nn.init.zeros_(layer.bias)

        self.audio_proj = self.audio_proj.to(device=self._embed_device, dtype=torch.float32)
        self.audio_prefix_pos = nn.Parameter(
            torch.zeros(1, self.audio_prefix_tokens, lm_dim, device=self._embed_device, dtype=torch.float32)
        )
        nn.init.normal_(self.audio_prefix_pos, mean=0.0, std=0.01)
        self.audio_gate = nn.Parameter(torch.tensor(0.0, device=self._embed_device, dtype=torch.float32))

    def _make_audio_prefix_tokens(self, audio_path: str) -> torch.Tensor:
        """Temporal chunking over wav2vec2 states -> K virtual audio prefix tokens."""
        hidden = self.audio_encoder.encode_sequence(audio_path)
        hidden = hidden.to(device=self._embed_device, dtype=torch.float32)  # (1, T, H)
        hidden = torch.nan_to_num(hidden, nan=0.0, posinf=1.0, neginf=-1.0)

        T = max(1, hidden.shape[1])
        edges = torch.linspace(0, T, steps=self.audio_prefix_tokens + 1, device=hidden.device).long()
        chunks = []
        for i in range(self.audio_prefix_tokens):
            start = int(edges[i].item())
            end = int(edges[i + 1].item())
            if end <= start:
                end = min(T, start + 1)
            seg = hidden[:, start:end, :]
            if seg.shape[1] == 0:
                seg = hidden[:, -1:, :]
            chunks.append(seg.mean(dim=1))

        chunk_vecs = torch.stack(chunks, dim=1)  # (1, K, H)
        chunk_vecs = F.normalize(chunk_vecs, dim=-1, eps=1e-6)

        audio_tok = self.audio_proj(chunk_vecs)
        audio_tok = audio_tok + self.audio_prefix_pos
        audio_tok = torch.tanh(audio_tok) * float(CFG.get("audio_prefix_scale", 0.08))
        audio_tok = torch.sigmoid(self.audio_gate) * audio_tok
        audio_tok = torch.nan_to_num(audio_tok, nan=0.0, posinf=0.1, neginf=-0.1)
        return audio_tok.to(self._embed_dtype)

    @staticmethod
    def _make_prefix(question: str) -> str:
        return f"Question: {question}\n\nThinking:\n"

    @staticmethod
    def _make_target(reasoning: str, answer: str) -> str:
        return f"{reasoning}\n\nFinal Answer:\n{answer}"

    def _prepare_training_inputs(
        self,
        audio_path: str,
        question: str,
        reasoning: str,
        answer: str,
    ):
        """Return (combined_embeds, combined_attn, labels, label_weights)."""
        audio_tok = self._make_audio_prefix_tokens(audio_path)

        prefix_str = self._make_prefix(question)
        target_str = self._make_target(reasoning, answer)
        reasoning_block = f"{reasoning}\n\nFinal Answer:\n"

        prefix_ids = self.tok(prefix_str, return_tensors="pt", add_special_tokens=True)["input_ids"]
        target_ids = self.tok(target_str, return_tensors="pt", add_special_tokens=False)["input_ids"]
        reasoning_ids = self.tok(reasoning_block, return_tensors="pt", add_special_tokens=False)["input_ids"]
        answer_start_in_target = reasoning_ids.shape[1]

        eos = torch.tensor([[self.tok.eos_token_id]], dtype=target_ids.dtype)
        target_ids = torch.cat([target_ids, eos], dim=1)

        input_ids = torch.cat([prefix_ids, target_ids], dim=1)
        max_text = CFG["max_seq_len"] - self.audio_prefix_tokens
        input_ids = input_ids[:, :max_text]

        prefix_len = min(prefix_ids.shape[1], input_ids.shape[1])

        input_ids = input_ids.to(self._embed_device)
        text_embeds = self.lm.get_input_embeddings()(input_ids)

        combined = torch.cat([audio_tok, text_embeds], dim=1)
        combined = torch.nan_to_num(combined, nan=0.0, posinf=1.0, neginf=-1.0)

        attn = torch.ones(combined.shape[:2], dtype=torch.long, device=combined.device)

        labels = input_ids.clone()
        labels[:, :prefix_len] = -100

        answer_start = prefix_len + answer_start_in_target
        label_weights = torch.ones_like(labels, dtype=torch.float32)
        label_weights[:, :prefix_len] = 0.0
        if answer_start < labels.shape[1]:
            label_weights[:, answer_start:] *= float(CFG.get("answer_loss_weight", 2.5))

        pad_label = torch.full((1, self.audio_prefix_tokens), -100, dtype=labels.dtype, device=labels.device)
        pad_weights = torch.zeros((1, self.audio_prefix_tokens), dtype=label_weights.dtype, device=label_weights.device)
        labels = torch.cat([pad_label, labels], dim=1)
        label_weights = torch.cat([pad_weights, label_weights], dim=1)

        assert (labels != -100).any(), (
            "All labels were masked — increase max_seq_len or shorten the question."
        )

        return combined, attn, labels, label_weights

    def _prepare_inference_inputs(self, audio_path: str, question: str):
        """Return (combined_embeds, combined_attn) for generation."""
        audio_tok = self._make_audio_prefix_tokens(audio_path)

        prefix_str = self._make_prefix(question)
        tokens = self.tok(prefix_str, return_tensors="pt", add_special_tokens=True)
        input_ids = tokens["input_ids"].to(self._embed_device)
        text_embeds = self.lm.get_input_embeddings()(input_ids)

        combined = torch.cat([audio_tok, text_embeds], dim=1)
        combined = torch.nan_to_num(combined, nan=0.0, posinf=1.0, neginf=-1.0)
        attn = torch.ones(combined.shape[:2], dtype=torch.long, device=combined.device)
        return combined, attn

    def forward(
        self,
        audio_path: str,
        question: str,
        reasoning: Optional[str] = None,
        answer: Optional[str] = None,
    ):
        if reasoning is not None and answer is not None:
            embeds, attn, labels, label_weights = self._prepare_training_inputs(
                audio_path, question, reasoning, answer
            )
            logits = self.lm(inputs_embeds=embeds, attention_mask=attn).logits

            shift_logits = logits[:, :-1, :].contiguous()
            shift_labels = labels[:, 1:].contiguous()
            shift_weights = label_weights[:, 1:].contiguous()

            token_loss = F.cross_entropy(
                shift_logits.view(-1, shift_logits.size(-1)),
                shift_labels.view(-1),
                reduction="none",
                ignore_index=-100,
            ).view_as(shift_labels)

            mask = (shift_labels != -100).float()
            effective_weights = shift_weights * mask
            denom = effective_weights.sum().clamp_min(1.0)
            loss = (token_loss * effective_weights).sum() / denom

            if not torch.isfinite(loss):
                raise FloatingPointError("Non-finite loss from LM forward pass")
            return type("TrainOutput", (object,), {"loss": loss, "logits": logits})()

        return self._prepare_inference_inputs(audio_path, question)


# Instantiate
model = AudioLanguageModel(AudioEncoder(), lm_model, tokenizer)
print(f"✓ AudioLanguageModel ready  |  projection params: {sum(p.numel() for p in model.audio_proj.parameters()):,}")

Loading weights:   0%|          | 0/210 [00:00<?, ?it/s]

Wav2Vec2Model LOAD REPORT from: facebook/wav2vec2-base-960h
Key               | Status     | 
------------------+------------+-
lm_head.weight    | UNEXPECTED | 
lm_head.bias      | UNEXPECTED | 
masked_spec_embed | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


  LLM embed device: cuda:0, dtype: torch.float32
✓ AudioLanguageModel ready  |  projection params: 8,529,920


## 7 · Training

In [28]:
def _sample_order_for_epoch(dataset: List[AudioSample], epoch: int, total_epochs: int) -> np.ndarray:
    """Simple curriculum: easier single-cue patterns first, then full mix."""
    warmup = int(CFG.get("curriculum_warmup_epochs", 3))
    if epoch > warmup:
        return np.random.permutation(len(dataset))

    easy_keywords = (
        "ascending", "descending", "increase", "decrease", "longer", "how many", "overlap", "pause"
    )
    easy_idx, hard_idx = [], []
    for i, s in enumerate(dataset):
        q = s.question.lower()
        if any(k in q for k in easy_keywords):
            easy_idx.append(i)
        else:
            hard_idx.append(i)

    easy_idx = np.random.permutation(easy_idx).tolist()
    hard_idx = np.random.permutation(hard_idx).tolist()
    if epoch == 1:
        return np.array(easy_idx, dtype=np.int64)
    return np.array(easy_idx + hard_idx, dtype=np.int64)


def train_model(
    model: AudioLanguageModel,
    dataset: List[AudioSample],
    num_epochs: int = 14,
    lr: float = 8e-6,
) -> AudioLanguageModel:
    """
    Supervised fine-tuning for temporal reasoning and ESC-50 hybrid grounding.
    Trains ONLY audio projection + LoRA adapters.
    Includes lightweight curriculum + wrong-audio contrastive regularization.
    """

    proj_params = list(model.audio_proj.parameters()) + [model.audio_prefix_pos, model.audio_gate]
    lora_params = [p for _, p in model.lm.named_parameters() if p.requires_grad]
    n_train = sum(p.numel() for p in proj_params + lora_params)
    print(f"Trainable parameters: {n_train:,}")

    optimizer = torch.optim.AdamW(
        [
            {"params": proj_params, "lr": lr * 1.5},
            {"params": lora_params, "lr": lr},
        ],
        weight_decay=CFG["weight_decay"],
        eps=1e-8,
    )

    warmup_steps = max(2, len(dataset))
    scheduler = torch.optim.lr_scheduler.LinearLR(
        optimizer,
        start_factor=0.2,
        end_factor=1.0,
        total_iters=warmup_steps,
    )

    model.train()
    best_loss = float("inf")
    patience, patience_limit = 0, 4
    global_step = 0

    for epoch in range(1, num_epochs + 1):
        epoch_loss = 0.0
        valid_steps = 0

        order = _sample_order_for_epoch(dataset, epoch, num_epochs)
        if len(order) == 0:
            print(f"  Epoch {epoch}: no curriculum samples.")
            break

        for idx in order:
            s = dataset[int(idx)]
            optimizer.zero_grad(set_to_none=True)

            try:
                pos_out = model(
                    audio_path=s.audio_path,
                    question=s.question,
                    reasoning=s.reasoning,
                    answer=s.answer,
                )
                pos_loss = pos_out.loss.float()

                loss = pos_loss
                if len(dataset) > 1 and np.random.rand() < float(CFG.get("neg_pair_prob", 0.5)):
                    neg_idx = int(np.random.randint(0, len(dataset) - 1))
                    if neg_idx >= idx:
                        neg_idx += 1
                    neg_audio = dataset[neg_idx].audio_path

                    neg_out = model(
                        audio_path=neg_audio,
                        question=s.question,
                        reasoning=s.reasoning,
                        answer=s.answer,
                    )
                    neg_loss = neg_out.loss.float()

                    margin = float(CFG.get("contrastive_margin", 0.25))
                    contrastive = F.relu(margin + pos_loss - neg_loss)
                    loss = loss + float(CFG.get("contrastive_weight", 0.35)) * contrastive

                gate_reg = 1e-3 * (torch.sigmoid(model.audio_gate) - 0.5).pow(2)
                loss = loss + gate_reg

            except Exception as e:
                print(f"  ⚠ Skipping sample {idx}: {e}")
                continue

            if not torch.isfinite(loss):
                print(f"  ⚠ Non-finite loss at epoch {epoch}, sample {idx} — skipping")
                continue

            loss.backward()
            torch.nn.utils.clip_grad_norm_(proj_params, max_norm=0.5)
            torch.nn.utils.clip_grad_norm_(lora_params, max_norm=CFG["grad_clip"])

            optimizer.step()
            if global_step < warmup_steps:
                scheduler.step()

            epoch_loss += loss.item()
            valid_steps += 1
            global_step += 1

        if valid_steps == 0:
            print(f"  Epoch {epoch}: no valid steps.")
            break

        avg = epoch_loss / valid_steps
        tag = ""
        if avg < best_loss - 1e-4:
            best_loss = avg
            patience = 0
            tag = " ★"
        else:
            patience += 1
            if patience >= patience_limit:
                print(f"  Early stop at epoch {epoch} (patience={patience_limit})")
                break

        gate_val = float(torch.sigmoid(model.audio_gate).detach().cpu())
        print(f"  Epoch {epoch:>2}/{num_epochs}  loss={avg:.4f}{tag}  audio_gate={gate_val:.3f}")

    model.eval()
    return model


print(
    f"Hybrid training set size: {len(train_dataset)} "
    f"(temporal={len(dataset)}, esc50={len(esc50_train)})"
)
model = train_model(model, train_dataset, num_epochs=CFG["num_epochs"], lr=CFG["lr"])

Hybrid training set size: 24 (temporal=12, esc50=1600)
Trainable parameters: 32,138,241
  Epoch  1/8  loss=2.4852 ★  audio_gate=0.500
  Epoch  2/8  loss=2.3876 ★  audio_gate=0.500
  Epoch  3/8  loss=1.9512 ★  audio_gate=0.500
  Epoch  4/8  loss=1.4784 ★  audio_gate=0.500
  Epoch  5/8  loss=1.1056 ★  audio_gate=0.500
  Epoch  6/8  loss=0.9032 ★  audio_gate=0.500
  Epoch  7/8  loss=0.7775 ★  audio_gate=0.500
  Epoch  8/8  loss=0.7031 ★  audio_gate=0.500


## 8 · Inference & Output Parsing

In [29]:
@torch.no_grad()
def generate_answer(
    model: AudioLanguageModel,
    audio_path: str,
    question: str,
    max_new_tokens: int = CFG["max_new_tokens"],
    temperature: float = CFG["temperature"],
    top_p: float = CFG["top_p"],
) -> str:
    """
    Single-pass generation.
    Returns the full 'Thinking: … \\n Final Answer: …' string.
    """
    model.eval()
    embeds, attn = model(audio_path, question)

    do_sample = temperature > 0
    gen_kwargs = dict(
        inputs_embeds=embeds,
        attention_mask=attn,
        max_new_tokens=max_new_tokens,
        do_sample=do_sample,
        pad_token_id=model.tok.pad_token_id,
        eos_token_id=model.tok.eos_token_id,
    )
    if do_sample:
        gen_kwargs["temperature"] = temperature
        gen_kwargs["top_p"] = top_p

    output_ids = model.lm.generate(**gen_kwargs)
    continuation = model.tok.decode(output_ids[0], skip_special_tokens=True)

    # The prompt already ends with "Thinking:\n", so the continuation is the
    # reasoning + final answer.  Reconstruct the full output for parsing.
    full = f"Thinking:\n{continuation}"
    return full


def parse_output(text: str) -> Dict[str, str]:
    """
    Parse the structured output into thinking / answer fields.
    Robust to minor formatting variations.
    """
    thinking, answer = "", ""

    if "Thinking:" in text:
        remainder = text.split("Thinking:", 1)[1]
        if "Final Answer:" in remainder:
            t_part, a_part = remainder.split("Final Answer:", 1)
            thinking = t_part.strip()
            answer = a_part.strip()
        else:
            thinking = remainder.strip()
    elif "Final Answer:" in text:
        answer = text.split("Final Answer:", 1)[1].strip()

    return {"thinking": thinking, "answer": answer}


# Quick inference test on the first sample
_test = dataset[0]
_out = generate_answer(model, _test.audio_path, _test.question)
print("── Sample inference ──")
print(_out)
print()
_parsed = parse_output(_out)
print(f"  Parsed thinking : {_parsed['thinking'][:80]}…")
print(f"  Parsed answer   : {_parsed['answer'][:80]}")

── Sample inference ──
Thinking:
The pitch starts low, then increases, and then decreases.

Final Answer:
Pitch increases and then decreases.

  Parsed thinking : The pitch starts low, then increases, and then decreases.…
  Parsed answer   : Pitch increases and then decreases.


## 9 · Evaluation

In [ ]:
def _normalize_text(x: str) -> str:
    x = x.lower().replace("_", " ").strip()
    x = re.sub(r"[^a-z0-9\s]", " ", x)
    return " ".join(x.split())


def _extract_esc50_label(text: str, labels: List[str]) -> str:
    norm = _normalize_text(text)
    if not norm:
        return ""

    # exact/contains first
    for label in labels:
        lnorm = _normalize_text(label)
        if lnorm and lnorm in norm:
            return label

    # token overlap fallback
    pred_toks = set(norm.split())
    best_label, best_score = "", 0.0
    for label in labels:
        lnorm = _normalize_text(label)
        ltoks = set(lnorm.split())
        if not ltoks:
            continue
        overlap = len(pred_toks & ltoks) / len(ltoks)
        if overlap > best_score:
            best_score = overlap
            best_label = label
    if best_score >= 0.5:
        return best_label
    return ""


def _stable_majority_vote(values: List[str]) -> str:
    """Deterministic majority vote with lexical tie-break for reproducible eval."""
    counts: Dict[str, int] = {}
    for v in values:
        counts[v] = counts.get(v, 0) + 1
    ranked = sorted(counts.items(), key=lambda kv: (-kv[1], kv[0]))
    return ranked[0][0] if ranked else ""


def _extract_yes_no(text: str) -> Optional[str]:
    t = _normalize_text(text)
    if not t:
        return None
    no_patterns = [
        "no", "not", "none", "does not", "dont", "do not", "cannot", "cant",
    ]
    yes_patterns = [
        "yes", "there is", "there are", "present", "occurs", "exists",
    ]
    if any(p in t for p in no_patterns):
        return "no"
    if any(p in t for p in yes_patterns):
        return "yes"
    return None


def _extract_trend(text: str) -> Optional[str]:
    t = _normalize_text(text)
    if any(k in t for k in ["ascending", "increase", "increasing", "rising", "upward"]):
        return "up"
    if any(k in t for k in ["descending", "decrease", "decreasing", "falling", "downward"]):
        return "down"
    return None


def _extract_count(text: str) -> Optional[int]:
    t = _normalize_text(text)
    m = re.search(r"\b(\d+)\b", t)
    if m:
        return int(m.group(1))
    word_to_num = {
        "one": 1, "two": 2, "three": 3, "four": 4, "five": 5,
    }
    for w, n in word_to_num.items():
        if re.search(rf"\b{w}\b", t):
            return n
    return None


def _semantic_answer_correct(question: str, ground_truth: str, predicted: str) -> bool:
    q = _normalize_text(question)
    gt = _normalize_text(ground_truth)
    pred = _normalize_text(predicted)
    if not pred:
        return False

    yn_gt = _extract_yes_no(gt)
    yn_pred = _extract_yes_no(pred)
    if any(k in q for k in ["longer", "overlap", "pause", "silent gap", "occur before"]):
        if yn_gt is not None and yn_pred is not None:
            return yn_gt == yn_pred

    if any(k in q for k in ["increase or decrease", "ascending or descending", "frequency trend", "sweep"]):
        gt_tr = _extract_trend(gt)
        pr_tr = _extract_trend(pred)
        if gt_tr is not None and pr_tr is not None:
            return gt_tr == pr_tr

    if "how many" in q or "how many distinct" in q or "how many discrete" in q:
        gt_n = _extract_count(gt)
        pr_n = _extract_count(pred)
        if gt_n is not None and pr_n is not None:
            return gt_n == pr_n

    if "frequency range" in q:
        range_keys = ["low", "mid", "high", "khz", "hz"]
        gt_has = {k for k in range_keys if k in gt}
        pred_has = {k for k in range_keys if k in pred}
        if gt_has and pred_has:
            return len(gt_has & pred_has) > 0

    return (gt in pred) or (pred in gt)


def evaluate(
    model: AudioLanguageModel,
    dataset: List[AudioSample],
    num_runs: int = CFG["eval_runs"],
) -> List[Dict]:
    """Evaluate standard dataset with stability + correctness metrics."""
    results: List[Dict] = []
    eval_temp = float(CFG.get("eval_temperature", 0.0))
    eval_top_p = float(CFG.get("eval_top_p", 1.0))
    semantic_match = bool(CFG.get("eval_semantic_match", True))

    for i, sample in enumerate(dataset):
        runs = []
        for _ in range(num_runs):
            raw = generate_answer(
                model,
                sample.audio_path,
                sample.question,
                temperature=eval_temp,
                top_p=eval_top_p,
            )
            runs.append(parse_output(raw))

        answers = [r["answer"] for r in runs]
        thinkings = [r["thinking"] for r in runs]

        answer_stable = len(set(answers)) == 1
        thinking_stable = len(set(thinkings)) <= 2

        majority_answer = _stable_majority_vote(answers)
        if semantic_match:
            answer_correct = _semantic_answer_correct(sample.question, sample.answer, majority_answer)
        else:
            gt = _normalize_text(sample.answer)
            pred = _normalize_text(majority_answer)
            answer_correct = (gt in pred) or (pred in gt)

        if answer_correct and thinking_stable:
            cat = "correct_reasoning_correct_answer"
        elif answer_correct and not thinking_stable:
            cat = "weak_reasoning_correct_answer"
        elif not answer_correct and thinking_stable:
            cat = "correct_reasoning_wrong_answer"
        else:
            cat = "weak_reasoning_wrong_answer"

        result = {
            "idx": i,
            "audio_path": sample.audio_path,
            "question": sample.question,
            "ground_truth": sample.answer,
            "predicted_answer": majority_answer,
            "predicted_thinking": runs[0]["thinking"],
            "answer_correct": answer_correct,
            "answer_stable": answer_stable,
            "thinking_stable": thinking_stable,
            "category": cat,
            "all_runs": runs,
        }
        results.append(result)

        status = "✓" if answer_correct else "✗"
        print(f"  [{status}] Q{i}: {sample.question[:60]}…  →  {cat}")

    n = len(results)
    n_correct = sum(r["answer_correct"] for r in results)
    n_stable = sum(r["answer_stable"] for r in results)
    print(f"\n  Accuracy  : {n_correct}/{n} ({100*n_correct/max(1,n):.0f}%)")
    print(f"  Stability : {n_stable}/{n} ({100*n_stable/max(1,n):.0f}%)")
    return results


def evaluate_audio_dependence(
    model: AudioLanguageModel,
    dataset: List[AudioSample],
) -> Dict[str, float]:
    """Measure dependency on audio by comparing true-audio vs shuffled-audio accuracy."""
    if len(dataset) < 2:
        return {"true_audio_accuracy": 0.0, "shuffled_audio_accuracy": 0.0, "audio_dependence_gap": 0.0}

    eval_temp = float(CFG.get("eval_temperature", 0.0))
    eval_top_p = float(CFG.get("eval_top_p", 1.0))
    semantic_match = bool(CFG.get("eval_semantic_match", True))

    true_correct = 0
    shuffled_correct = 0
    shuffled_paths = [dataset[(i + 1) % len(dataset)].audio_path for i in range(len(dataset))]

    for i, sample in enumerate(dataset):
        raw_true = generate_answer(
            model,
            sample.audio_path,
            sample.question,
            temperature=eval_temp,
            top_p=eval_top_p,
        )
        pred_true = parse_output(raw_true).get("answer", "")
        if semantic_match:
            true_correct += int(_semantic_answer_correct(sample.question, sample.answer, pred_true))
        else:
            gt = _normalize_text(sample.answer)
            pred = _normalize_text(pred_true)
            true_correct += int((gt in pred) or (pred in gt))

        raw_shuf = generate_answer(
            model,
            shuffled_paths[i],
            sample.question,
            temperature=eval_temp,
            top_p=eval_top_p,
        )
        pred_shuf = parse_output(raw_shuf).get("answer", "")
        if semantic_match:
            shuffled_correct += int(_semantic_answer_correct(sample.question, sample.answer, pred_shuf))
        else:
            gt = _normalize_text(sample.answer)
            pred = _normalize_text(pred_shuf)
            shuffled_correct += int((gt in pred) or (pred in gt))

    n = len(dataset)
    true_acc = true_correct / n
    shuf_acc = shuffled_correct / n
    gap = true_acc - shuf_acc

    print(f"\n  Audio dependence: true={100*true_acc:.1f}%  shuffled={100*shuf_acc:.1f}%  gap={100*gap:.1f} pts")
    return {
        "true_audio_accuracy": true_acc,
        "shuffled_audio_accuracy": shuf_acc,
        "audio_dependence_gap": gap,
    }


def evaluate_mmar_benchmark(
    model: AudioLanguageModel,
    benchmark: List[AudioSample],
) -> Dict[str, object]:
    """Evaluate on MMAR benchmark split (single deterministic run per item)."""
    eval_temp = float(CFG.get("eval_temperature", 0.0))
    eval_top_p = float(CFG.get("eval_top_p", 1.0))
    semantic_match = bool(CFG.get("eval_semantic_match", True))

    rows = []
    correct = 0

    for i, sample in enumerate(benchmark):
        raw = generate_answer(
            model,
            sample.audio_path,
            sample.question,
            max_new_tokens=96,
            temperature=eval_temp,
            top_p=eval_top_p,
        )
        parsed = parse_output(raw)
        pred_txt = parsed.get("answer", "")
        if semantic_match:
            ok = _semantic_answer_correct(sample.question, sample.answer, pred_txt)
        else:
            pred = _normalize_text(pred_txt)
            gt = _normalize_text(sample.answer)
            ok = (gt in pred) or (pred in gt)
        correct += int(ok)

        rows.append({
            "idx": i,
            "question": sample.question,
            "ground_truth": sample.answer,
            "prediction": pred_txt,
            "correct": ok,
        })

        mark = "✓" if ok else "✗"
        print(f"  [{mark}] MMAR-{i}: {sample.question[:58]}…")

    total = len(benchmark)
    accuracy = correct / max(1, total)
    report = {
        "benchmark": "MMAR",
        "num_samples": total,
        "num_correct": correct,
        "accuracy": accuracy,
        "rows": rows,
    }

    print(f"\n  MMAR Accuracy: {correct}/{total} ({100*accuracy:.1f}%)")
    return report


def evaluate_esc50(
    model: AudioLanguageModel,
    esc50_val: List[AudioSample],
    labels: List[str],
) -> Dict[str, object]:
    """Evaluate ESC-50 validation split as class prediction via generated answer text."""
    if len(esc50_val) == 0 or len(labels) == 0:
        print("\n── Evaluation (ESC-50) ──")
        print("  ESC-50 evaluation skipped (no validation samples loaded).")
        return {
            "benchmark": "ESC-50",
            "num_samples": 0,
            "num_correct": 0,
            "accuracy": 0.0,
            "rows": [],
            "per_class": {},
        }

    eval_temp = float(CFG.get("eval_temperature", 0.0))
    eval_top_p = float(CFG.get("eval_top_p", 1.0))
    eval_limit = int(CFG.get("esc50_eval_limit", 80))
    esc50_gen_tokens = int(CFG.get("esc50_eval_max_new_tokens", 24))

    eval_subset = esc50_val[:eval_limit] if eval_limit > 0 else esc50_val

    rows = []
    label_norm_map = {_normalize_text(label): label for label in labels}
    class_totals = {label_norm: 0 for label_norm in label_norm_map.keys()}
    class_correct = {label_norm: 0 for label_norm in label_norm_map.keys()}
    correct = 0

    print("\n── Evaluation (ESC-50) ──")
    print(f"  Using {len(eval_subset)}/{len(esc50_val)} validation samples")
    for i, sample in enumerate(eval_subset):
        raw = generate_answer(
            model,
            sample.audio_path,
            sample.question,
            max_new_tokens=esc50_gen_tokens,
            temperature=eval_temp,
            top_p=eval_top_p,
        )
        parsed = parse_output(raw)
        answer_text = parsed.get("answer", "") or raw
        pred_label = _extract_esc50_label(answer_text, labels)
        pred_label_norm = _normalize_text(pred_label)
        gt_label = _normalize_text(sample.answer)
        ok = pred_label_norm == gt_label

        class_totals[gt_label] = class_totals.get(gt_label, 0) + 1
        class_correct[gt_label] = class_correct.get(gt_label, 0) + int(ok)
        correct += int(ok)

        rows.append({
            "idx": i,
            "audio_path": sample.audio_path,
            "ground_truth": gt_label,
            "predicted_label": pred_label_norm,
            "predicted_label_raw": pred_label,
            "raw_answer": answer_text,
            "correct": ok,
        })

        mark = "✓" if ok else "✗"
        print(f"  [{mark}] ESC50-{i}: gt={gt_label} pred={pred_label_norm or 'unknown'}")

    total = len(eval_subset)
    acc = correct / max(1, total)
    per_class = {}
    for label_norm, n in class_totals.items():
        if n > 0:
            label_name = label_norm_map.get(label_norm, label_norm)
            per_class[label_name] = class_correct.get(label_norm, 0) / n

    print(f"\n  ESC-50 Accuracy: {correct}/{total} ({100*acc:.1f}%)")
    return {
        "benchmark": "ESC-50",
        "num_samples": total,
        "num_correct": correct,
        "accuracy": acc,
        "rows": rows,
        "per_class": per_class,
        "eval_limit": eval_limit,
        "full_val_size": len(esc50_val),
    }


print("── Evaluation (training scenes) ──")
eval_results = evaluate(model, dataset, num_runs=2)
audio_dep_report = evaluate_audio_dependence(model, dataset)

print("\n── Evaluation (MMAR benchmark) ──")
mmar_report = evaluate_mmar_benchmark(model, mmar_benchmark)
mmar_report["audio_dependence"] = audio_dep_report

esc50_report = evaluate_esc50(model, esc50_val, ESC50_LABELS)

combined_report = {
    "temporal_scene": {
        "num_samples": len(eval_results),
        "num_correct": int(sum(r["answer_correct"] for r in eval_results)),
        "accuracy": float(sum(r["answer_correct"] for r in eval_results) / max(1, len(eval_results))),
    },
    "mmar": {
        "num_samples": mmar_report.get("num_samples", 0),
        "num_correct": mmar_report.get("num_correct", 0),
        "accuracy": mmar_report.get("accuracy", 0.0),
    },
    "esc50": {
        "num_samples": esc50_report.get("num_samples", 0),
        "num_correct": esc50_report.get("num_correct", 0),
        "accuracy": esc50_report.get("accuracy", 0.0),
    },
}
print("\n── Combined summary ──")
print(json.dumps(combined_report, indent=2))

── Evaluation (training scenes) ──
  [✓] Q0: Does pitch increase or decrease over time?…  →  correct_reasoning_correct_answer
  [✗] Q1: Does pitch increase or decrease over time?…  →  correct_reasoning_wrong_answer
  [✗] Q2: Is this an ascending or descending sweep?…  →  correct_reasoning_wrong_answer
  [✗] Q3: Is this an ascending or descending sweep?…  →  correct_reasoning_wrong_answer
  [✓] Q4: How many distinct tone events are present?…  →  correct_reasoning_correct_answer
  [✓] Q5: Is the first tone longer than the second?…  →  correct_reasoning_correct_answer
  [✗] Q6: Is the first tone longer than the second?…  →  correct_reasoning_wrong_answer
  [✗] Q7: Is there a period where two tones are sounding at the same t…  →  correct_reasoning_wrong_answer
  [✓] Q8: Is there a noticeable silent gap between two tone events?…  →  correct_reasoning_correct_answer
  [✗] Q9: What frequency range best describes this tone?…  →  correct_reasoning_wrong_answer
  [✓] Q10: Did the lower-pitched e

## 10 · JSONL Output

In [ ]:
def save_results_jsonl(
    results: List[Dict],
    path: str = CFG["output_path"],
) -> None:
    """Write results in JSONL format (one JSON object per line)."""
    with open(path, "w", encoding="utf-8") as f:
        for r in results:
            entry = {
                "id": f"sample_{r['idx']:04d}",
                "thinking_prediction": r["predicted_thinking"],
                "answer_prediction": r["predicted_answer"],
                "answer_correct": r["answer_correct"],
            }
            f.write(json.dumps(entry, ensure_ascii=False) + "\n")
    print(f"✓ Saved {len(results)} scene-eval results → {path}")


def save_mmar_report_json(path: str = "./mmar_report.json") -> None:
    out = Path(path).resolve()
    with open(out, "w", encoding="utf-8") as f:
        json.dump(mmar_report, f, indent=2, ensure_ascii=False)
    print(f"✓ Saved MMAR report → {out}")


def save_esc50_report_json(path: str = "./esc50_report.json") -> None:
    out = Path(path).resolve()
    with open(out, "w", encoding="utf-8") as f:
        json.dump(esc50_report, f, indent=2, ensure_ascii=False)
    print(f"✓ Saved ESC-50 report → {out}")


def save_combined_report_json(path: str = "./combined_report.json") -> None:
    out = Path(path).resolve()
    with open(out, "w", encoding="utf-8") as f:
        json.dump(combined_report, f, indent=2, ensure_ascii=False)
    print(f"✓ Saved combined report → {out}")


save_results_jsonl(eval_results)
save_mmar_report_json()
save_esc50_report_json()
save_combined_report_json()

print("\n── results.jsonl ──")
with open(CFG["output_path"], encoding="utf-8") as f:
    for line in f:
        print(line.rstrip())

## Optional · Save / Load Checkpoint

In [ ]:
def save_checkpoint(model: AudioLanguageModel, path: str = CFG["ckpt_dir"]):
    """Save the trainable components (LoRA adapter + projection layer)."""
    os.makedirs(path, exist_ok=True)
    # LoRA adapter
    model.lm.save_pretrained(os.path.join(path, "lora_adapter"))
    # Projection layer
    torch.save(
        model.audio_proj.state_dict(),
        os.path.join(path, "audio_proj.pt"),
    )
    print(f"✓ Checkpoint saved → {path}/")


def load_checkpoint(model: AudioLanguageModel, path: str = CFG["ckpt_dir"]):
    """Reload trainable components from disk."""
    from peft import PeftModel
    proj_path = os.path.join(path, "audio_proj.pt")
    model.audio_proj.load_state_dict(torch.load(proj_path, map_location="cpu"))
    print(f"✓ Checkpoint loaded ← {path}/")
    return model


save_checkpoint(model)

---
**Done.** The full pipeline — data → training → inference → evaluation → JSONL
output — is complete. To use your own audio data, replace the synthetic dataset
in Section 4 with real `AudioSample` entries pointing to `.wav` files.